<a href="https://colab.research.google.com/github/Askobarus/nlp_course/blob/2024/week05_transfer/homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Load data and model

In [ ]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

### Tokenize the data

In [ ]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

In [ ]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model = model.to(device)

cuda


In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

true_values = []
pred_values = []
for batch in tqdm(val_loader):
    # here be your training code
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
      predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids'])

    pred = torch.argmax(predicted.logits, dim=1).data

    true_values += list(batch['labels'].data.cpu().numpy())
    pred_values += list(pred.cpu().numpy())
print("Sample batch:", batch)

accuracy = accuracy_score(true_values, pred_values)


100%|██████████| 2527/2527 [05:07<00:00,  8.23it/s]

Sample batch: {'labels': tensor([0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0'), 'idx': tensor([40416, 40417, 40418, 40419, 40420, 40421, 40422, 40423, 40424, 40425,
        40426, 40427, 40428, 40429], device='cuda:0'), 'input_ids': tensor([[ 101, 1327, 1110,  ...,    0,    0,    0],
        [ 101, 1327, 1132,  ...,    0,    0,    0],
        [ 101, 1327, 1110,  ...,    0,    0,    0],
        ...,
        [ 101, 2181, 2903,  ...,    0,    0,    0],
        [ 101, 1731, 1202,  ...,    0,    0,    0],
        [ 101, 1731, 1169,  ...,    0,    0,    0]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ...

In [ ]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [73]:
# Load the model and tokenizer
model_name = "microsoft/deberta-v3-small"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
#model = transformers.AutoModelForSequenceClassification.from_pretrained(path + 'deberta-v3-lora-qqp')
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [74]:
# Подготовка данных для обучения
train_set = qqp_preprocessed['train']
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=16, shuffle=True, collate_fn=transformers.default_data_collator,
    num_workers=2
)

# Валидационный загрузчик (оставляем как было)
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

In [75]:
# Настройка LoRA для DeBERTa-v3
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=6,  # rank
    lora_alpha=24,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"]  # Для DeBERTa-v3
)

# Применяем LoRA к модели
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

# Настройка аргументов обучения
training_args = transformers.TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",  # или "tensorboard", "wandb"
    remove_unused_columns=False,  # Важно для PEFT!
    fp16=True,  # Включить если используете GPU с поддержкой FP16
    dataloader_pin_memory=False,
)

# Создание Trainer и запуск обучения
trainer = transformers.Trainer(
     model=lora_model,
     args=training_args,
     train_dataset= qqp_preprocessed['train'],
     eval_dataset= qqp_preprocessed['validation'],
)

# Оптимизатор только для обучаемых параметров
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

trainable params: 112,130 || all params: 142,008,580 || trainable%: 0.0790


In [ ]:
# 7. Запуск обучения
print("Начинаем обучение...")
trainer.train()

# 8. Сохранение модели
trainer.save_model("./lora-deberta-final")
tokenizer.save_pretrained("./lora-deberta-final")

# Также можно сохранить только адаптеры
model.save_pretrained("./lora-adapters-only")

Начинаем обучение...


TypeError: DebertaV2ForSequenceClassification.forward() got an unexpected keyword argument 'idx'

In [76]:
# Цикл обучения
num_epochs = 1
for epoch in range(num_epochs):
    # Обучение
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        batch_on_dev = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(
            input_ids=batch_on_dev['input_ids'],
            attention_mask=batch_on_dev['attention_mask'],
            token_type_ids=batch_on_dev['token_type_ids'],
            labels=batch_on_dev['labels']
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Валидация
    model.eval()
    true_values = []
    pred_values = []
    val_loss = 0

    for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids'],
                labels=batch['labels']
            )

            predicted = outputs.logits
            pred = torch.argmax(predicted, dim=1).data

            true_values += list(batch['labels'].data.cpu().numpy())
            pred_values += list(pred.cpu().numpy())
            val_loss += outputs.loss.item()

    # Вычисление метрик
    train_loss_avg = train_loss / len(train_loader)
    val_loss_avg = val_loss / len(val_loader)
    accuracy = accuracy_score(true_values, pred_values)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss_avg:.4f}")
    print(f"Val Loss: {val_loss_avg:.4f}")
    print(f"Val Accuracy: {accuracy:.4f}")
    print("-" * 50)

Epoch 1 Validation: 100%|██████████| 2527/2527 [03:14<00:00, 13.01it/s]

Epoch 1/1
Train Loss: 0.4819
Val Loss: 0.4432
Val Accuracy: 0.7800
--------------------------------------------------


In [77]:
# Финальная валидация после обучения
model.eval()
true_values = []
pred_values = []

for batch in tqdm(val_loader, desc="Final Validation"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids']
        )

        pred = torch.argmax(outputs.logits, dim=1).data
        true_values += list(batch['labels'].data.cpu().numpy())
        pred_values += list(pred.cpu().numpy())

final_accuracy = accuracy_score(true_values, pred_values)
print(f"Final Validation Accuracy: {final_accuracy:.4f}")

# Сохранение модели LoRA
model.save_pretrained(path + "deberta-v3-lora-qqp")

Final Validation: 100%|██████████| 2527/2527 [03:13<00:00, 13.03it/s]


Final Validation Accuracy: 0.7800


### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.